## Imports, Path definitions

In [1]:
import socket, os
print(socket.gethostname())   # shows which machine/node
print(os.getcwd())             # shows working directory


LAPTOP-RLFB5OCF
/home/lois/pypsa-eur/pypsa-eur/scripts/_testing


In [1]:
import pypsa
import numpy as np
import pandas as pd
import xarray as xr
import atlite
import yaml
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# directory set up on cluster

# root = Path.cwd()
# pypsa_eur_root = root  
# home_root = root.parent        

# pypsa_dmg_root = home_root / "pypsa-damage"
# pypsa_eur_scripts = pypsa_eur_root / "scripts"

# sys.path.insert(0, str(pypsa_eur_scripts / "_testing"))

# directory set up on local machine
root = Path.cwd()
pypsa_eur_root = root.parent.parent  
home_root = pypsa_eur_root.parent        

pypsa_dmg_root = home_root / "pypsa-damage"
pypsa_eur_scripts = pypsa_eur_root / "scripts"

sys.path.insert(0, str(pypsa_eur_root))
sys.path.insert(0, str(pypsa_eur_scripts))
sys.path.insert(0, str(pypsa_eur_scripts / "_testing"))

In [2]:
sys.path

['/home/lois/pypsa-eur/pypsa-eur/scripts/_testing',
 '/home/lois/pypsa-eur/pypsa-eur/scripts',
 '/home/lois/pypsa-eur/pypsa-eur',
 '/home/lois/pypsa-eur/pypsa-eur/.pixi/envs/default/lib/python313.zip',
 '/home/lois/pypsa-eur/pypsa-eur/.pixi/envs/default/lib/python3.13',
 '/home/lois/pypsa-eur/pypsa-eur/.pixi/envs/default/lib/python3.13/lib-dynload',
 '',
 '/home/lois/pypsa-eur/pypsa-eur/.pixi/envs/default/lib/python3.13/site-packages']

### Config name

In [3]:
# ====================
# CONFIGURATION
# ====================

# ---- Single-scenario (for network analysis in Cells 5-8) ----
config_name = "2022-FR"

# ---- Multi-scenario CF comparison (FR-summer_years+dmg) ----
CONFIG_PATH       = pypsa_eur_root / "config/config.weather_years+dmg.yaml"
DMG_CONFIG_PATH   = pypsa_eur_root / "config/damage_config.yaml"
CLUSTERS          = 5
CF_ACTUAL_DIR     = pypsa_dmg_root / "historic_comparison/outputs"
CF_ACTUAL_PATTERN = "cf_B14_plant_{year}-01-01_{year}-12-31.csv"
cf_fixed_factor   = 0.616
SCENARIO_GLOB     = "weather_year_*_dmg_cap"  # change to *_dmg_dis for dispatch-damage scenarios

### Network definitions

In [4]:
#elec_base_path = f"{pypsa_eur_root}/results/{config_name}/{config_name}-base/networks/base_s_5_elec_.nc" 
elec_damaged_cap_path = f"{pypsa_eur_root}/results/{config_name}/{config_name}-nucl_dmg_cap/networks/base_s_5_elec_.nc" 
elec_damaged_dis_path = f"{pypsa_eur_root}/results/{config_name}/{config_name}-nucl_dmg_dis/networks/base_s_5_elec__damaged-dispatch.nc" 
elec_damaged_base_dis_path = f"{pypsa_eur_root}/results/{config_name}/{config_name}-nucl_dmg_dis/networks/base_s_5_elec_.nc" 

#n_b = pypsa.Network(elec_base_path)
n_dc = pypsa.Network(elec_damaged_cap_path)
n_dd = pypsa.Network(elec_damaged_dis_path)
n_bd = pypsa.Network(elec_damaged_base_dis_path)

INFO:pypsa.network.io:New version 1.2.2 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.2 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.2 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


### Statistics setup

In [5]:
print("Available groupers:", list(pypsa.statistics.groupers.list_groupers().keys()))
grouper = ["bus", "carrier"]

Available groupers: ['bus', 'bus_carrier', 'carrier', 'country', 'location', 'name', 'unit']


In [6]:
cf_actual = pd.read_csv(
    pypsa_dmg_root / "historic_comparison/outputs/cf_B14_plant_2022-01-01_2022-12-31.csv",
    index_col=0,
    parse_dates=True,
)
cf_actual.head()


,country,unit_name,unit_code,psr_type,generation_mw,installed_capacity_mw,capacity_factor
timestamp,,,,,,,
2022-01-01 00:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1263.0,1310,0.964122
2022-01-01 01:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1262.0,1310,0.963359
2022-01-01 02:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1261.0,1310,0.962595
2022-01-01 03:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1261.0,1310,0.962595
2022-01-01 04:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1264.0,1310,0.964885


In [7]:
cf_actual.unit_name.unique()

array(['BELLEVILLE 1', 'BELLEVILLE 2', 'BLAYAIS 1', 'BLAYAIS 2',
       'BLAYAIS 3', 'BLAYAIS 4', 'BUGEY 2', 'BUGEY 3', 'BUGEY 4',
       'BUGEY 5', 'CATTENOM 1', 'CATTENOM 3', 'CATTENOM 4', 'CHINON 2',
       'CHINON 3', 'CHINON 4', 'CHOOZ 1', 'CIVAUX 1', 'CIVAUX 2',
       'CRUAS 1', 'CRUAS 3', 'CRUAS 4', 'DAMPIERRE 2', 'DAMPIERRE 3',
       'DAMPIERRE 4', 'FESSENHEIM 1', 'FLAMANVILLE 1', 'FLAMANVILLE 2',
       'GOLFECH 1', 'GOLFECH 2', 'GRAVELINES 2', 'GRAVELINES 3',
       'GRAVELINES 4', 'GRAVELINES 5', 'GRAVELINES 6', 'NOGENT 1',
       'NOGENT 2', 'PALUEL 1', 'PALUEL 2', 'PALUEL 3', 'PALUEL 4',
       'PENLY 2', 'ST ALBAN 1', 'ST ALBAN 2', 'ST LAURENT 2',
       'TRICASTIN 1', 'TRICASTIN 2', 'TRICASTIN 3', 'TRICASTIN 4',
       'CHINON 1', 'ST LAURENT 1', 'CRUAS 2', 'CATTENOM 2', 'CHOOZ 2',
       'GRAVELINES 1', 'DAMPIERRE 1', 'FESSENHEIM 2', 'PENLY 1'],
      dtype=object)

In [8]:
import re
from scripts.build_damage_profiles.build_nuclear_damage_profiles import load_nuclear_plants
from plot_damage import plot_plant_profile

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
with open(DMG_CONFIG_PATH) as f:
    dmg_cfg = yaml.safe_load(f)

_prefix           = cfg["run"].get("prefix", "")
SCENARIO_BASE_DIR = pypsa_eur_root / "resources" / _prefix


def _base_name(name: str) -> str:
    """Uppercase name with trailing unit number stripped (e.g. 'Paluel 2' → 'PALUEL')."""
    return re.sub(r"\s+\d+$", "", name.strip().upper())


def _build_mapping(powerplants_df: pd.DataFrame, cf_actual: pd.DataFrame) -> pd.DataFrame:
    unit_names = cf_actual.unit_name.unique()
    rows = []
    for pp_name in powerplants_df["Name"]:
        base    = _base_name(pp_name)
        matched = [u for u in unit_names if u.startswith(base)]
        for unit in matched:
            rows.append({"Name": pp_name, "unit_name": unit})
    return pd.DataFrame(rows).sort_values("Name").reset_index(drop=True)


def load_scenario(scenario_dir: Path) -> dict:
    """Load all data for one weather_year_* scenario directory."""
    year = int(re.search(r"weather_year_(\d+)_dmg", scenario_dir.name).group(1))

    powerplants_df = load_nuclear_plants(scenario_dir / f"powerplants_s_{CLUSTERS}.csv")

    plant_nc  = scenario_dir / "damage_profiles" / "nuclear_damage_plants.nc"
    damage_df = xr.open_dataset(plant_nc)["profile"].to_pandas()

    cf_actual = pd.read_csv(
        CF_ACTUAL_DIR / CF_ACTUAL_PATTERN.format(year=year),
        index_col=0, parse_dates=True,
    )
    mapping = _build_mapping(powerplants_df, cf_actual)

    return {
        "year":           year,
        "scenario_dir":   scenario_dir,
        "damage_df":      damage_df,
        "powerplants_df": powerplants_df,
        "cf_actual":      cf_actual,
        "mapping":        mapping,
    }


# Discover and load all matching scenarios
scenario_dirs = sorted(SCENARIO_BASE_DIR.glob(SCENARIO_GLOB))
scenarios     = [load_scenario(d) for d in scenario_dirs]

# Expose first scenario's data at module level (used by single-scenario cells below)
powerplants_df = scenarios[0]["powerplants_df"]
damage_df      = scenarios[0]["damage_df"]
cf_actual      = scenarios[0]["cf_actual"]
mapping        = scenarios[0]["mapping"]

print(f"Found {len(scenarios)} scenarios: {[s['year'] for s in scenarios]}")
print(f"Buses present: {sorted(powerplants_df['bus'].unique())}")

IndexError: list index out of range

In [18]:
scenario_dirs

[PosixPath('/home/lois/pypsa-eur/pypsa-eur/resources/FR-weather_years+dmg/weather_year_2018_dmg_cap'),
 PosixPath('/home/lois/pypsa-eur/pypsa-eur/resources/FR-weather_years+dmg/weather_year_2019_dmg_cap'),
 PosixPath('/home/lois/pypsa-eur/pypsa-eur/resources/FR-weather_years+dmg/weather_year_2020_dmg_cap'),
 PosixPath('/home/lois/pypsa-eur/pypsa-eur/resources/FR-weather_years+dmg/weather_year_2021_dmg_cap'),
 PosixPath('/home/lois/pypsa-eur/pypsa-eur/resources/FR-weather_years+dmg/weather_year_2022_dmg_cap')]

In [9]:
cf_actual

,country,unit_name,unit_code,psr_type,generation_mw,installed_capacity_mw,capacity_factor
timestamp,,,,,,,
2022-01-01 00:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1263.0,1310,0.964122
2022-01-01 01:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1262.0,1310,0.963359
2022-01-01 02:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1261.0,1310,0.962595
2022-01-01 03:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1261.0,1310,0.962595
2022-01-01 04:00:00+01:00,FR,BELLEVILLE 1,17W100P100P0090I,B14,1264.0,1310,0.964885
...,...,...,...,...,...,...,...
2022-12-08 18:00:00+01:00,FR,PENLY 1,17W100P100P0143N,B14,0.0,1330,0.000000
2022-12-08 19:00:00+01:00,FR,PENLY 1,17W100P100P0143N,B14,0.0,1330,0.000000
2022-12-08 20:00:00+01:00,FR,PENLY 1,17W100P100P0143N,B14,0.0,1330,0.000000


In [11]:
_data_cutout = cfg["data"]["cutout"]
_default_cutout = cfg["atlite"]["default_cutout"]

for s in scenarios:
    year = s["year"]
    cutout_name = re.sub(r"(?<=-)\d{4}(?=-)", str(year), _default_cutout)
    cutout_path = (
        pypsa_eur_root
        / "data/cutout"
        / _data_cutout["source"]
        / _data_cutout["version"]
        / (cutout_name + ".nc")
    )
    s["cutout_data"] = atlite.Cutout(path=cutout_path).data

cutout_data_by_year = {s["year"]: s["cutout_data"] for s in scenarios}
cutout_data = scenarios[0]["cutout_data"]  # backward compat for single-scenario cells

In [12]:
cutout_data

<xarray.Dataset> Size: 19GB
Dimensions:           (y: 157, x: 189, time: 8760)
Coordinates:
  * y                 (y) float64 1kB 33.0 33.25 33.5 33.75 ... 71.5 71.75 72.0
    lat               (y) float64 1kB dask.array<chunksize=(157,), meta=np.ndarray>
  * x                 (x) float64 2kB -12.0 -11.75 -11.5 ... 34.5 34.75 35.0
    lon               (x) float64 2kB dask.array<chunksize=(189,), meta=np.ndarray>
  * time              (time) datetime64[ns] 70kB 2018-01-01 ... 2018-12-31T23...
Data variables: (12/15)
    height            (y, x) float32 119kB dask.array<chunksize=(157, 189), meta=np.ndarray>
    wnd100m           (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    wnd_azimuth       (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    roughness         (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    influx_toa        (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    influx_direct     (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    ...                ...
    solar_azimuth     (time, y, x) float64 2GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    temperature       (time, y, x) float64 2GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    soil temperature  (time, y, x) float64 2GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    runoff            (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    wnd_gust10m       (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
    lake_s_temp       (time, y, x) float32 1GB dask.array<chunksize=(100, 157, 189), meta=np.ndarray>
Attributes:
    module:                  era5
    prepared_features:       ['height', 'temperature', 'influx', 'lake_s_temp...
    chunksize_time:          100
    Conventions:             CF-1.7
    history:                 2026-03-27T10:06 GRIB to CDM+CF via cfgrib-0.9.1...
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    institution:             European Centre for Medium-Range Weather Forecasts

In [16]:
powerplants_df

,Name,lat,lon,Capacity,bus
38,Nogent,48.5171,3.51810,2620.0,FR0 1
39,Paluel 2,49.8582,0.63540,5320.0,FR0 1
40,Penly 3,49.9764,1.21070,2660.0,FR0 1
41,St Alban,45.4043,4.75540,2670.0,FR0 3
42,Dampierre 4,47.7321,2.51850,3560.0,FR0 1
43,Belleville 5,47.5103,2.87501,2620.0,FR0 1
44,Gravelines 6,51.0141,2.13320,5460.0,FR0 1
45,Bugey 2,45.7973,5.27060,3580.0,FR0 3
46,Cattenom,49.4160,6.21690,5200.0,FR0 0
47,Chinon,47.2254,0.16560,3620.0,FR0 4


In [13]:
mapping

,Name,unit_name
0,Belleville 5,BELLEVILLE 1
1,Belleville 5,BELLEVILLE 2
2,Bugey 2,BUGEY 2
3,Bugey 2,BUGEY 3
4,Bugey 2,BUGEY 4
5,Bugey 2,BUGEY 5
6,Cattenom,CATTENOM 1
7,Cattenom,CATTENOM 2
8,Cattenom,CATTENOM 3
9,Cattenom,CATTENOM 4


In [14]:
cf_fixed_factor = 0.616

In [15]:
damage_df.head()

plant,Nogent,Paluel 2,Penly 3,St Alban,Dampierre 4,Belleville 5,Gravelines 6,Bugey 2,Cattenom,Chinon,Chooz 2,Civaux,Cruas 3,Flamanville 2,Golfech 2
time,,,,,,,,,,,,,,,
2018-06-01 00:00:00,1.0,1.0,1.0,1.0,0.997,1.0,0.997,1.0,0.991,1.0,1.0,1.0,1.0,1.0,1.0
2018-06-01 01:00:00,1.0,1.0,1.0,1.0,0.997,1.0,0.997,1.0,0.991,1.0,1.0,1.0,1.0,1.0,1.0
2018-06-01 02:00:00,1.0,1.0,1.0,1.0,1.000,1.0,0.997,1.0,0.991,1.0,1.0,1.0,1.0,1.0,1.0
2018-06-01 03:00:00,1.0,1.0,1.0,1.0,1.000,1.0,0.997,1.0,0.997,1.0,1.0,1.0,1.0,1.0,1.0
2018-06-01 04:00:00,1.0,1.0,1.0,1.0,1.000,1.0,0.997,1.0,0.997,1.0,1.0,1.0,1.0,1.0,1.0


In [16]:
plants = ['St Alban', 'Bugey', 'Golfech']

## Plot_plant_cf_comparison

In [ ]:
from plot_cf_temp import (
    plot_plant_cf_comparison,
    plot_plant_cf_comparison_mpl,
    build_cf_sources,
    build_multi_scenario_sources,
    cf_to_temp,
    cf_to_temp_max,
    cf_dev_to_temp_p90,
    cf_change_to_temp_max,
)
from _plotting_helpers import _resolve_date_range


In [20]:
s2022 = next(s for s in scenarios if s["year"] == 2022)
fig = plot_plant_cf_comparison(
    "St Alban", ("2022-06-01", "2022-08-31"),
    s2022["damage_df"], s2022["cf_actual"], s2022["mapping"],
)
fig.show()


In [21]:
fig = plot_plant_cf_comparison(
    "St Alban", ("07-01", "07-31"),
    scenarios=scenarios,
)
fig.show()



NameError: name '_PLOTLY_COLORS' is not defined

## Plot_cf_to_temp

In [ ]:
# Convenience variables for single-scenario (2022) usage cells below
_s = next(s for s in scenarios if s["year"] == 2022)


In [ ]:
sources = build_cf_sources(_s["damage_df"], _s["cf_actual"], _s["mapping"], cf_fixed_factor=cf_fixed_factor)

fig = cf_to_temp(cutout_data_by_year[2022], _s["powerplants_df"], sources,
                 plant_name="St Alban", date_range=("2022-07-01", "2022-07-31"))
fig.show()


In [ ]:
sources = build_cf_sources(_s["damage_df"], _s["cf_actual"], _s["mapping"], cf_fixed_factor=cf_fixed_factor)

for plant in _s["damage_df"].columns:
    print(f"{plant}")
    fig = cf_to_temp(cutout_data_by_year[2022], _s["powerplants_df"], sources,
                     plant_name=plant, date_range=("2022-07-01", "2022-07-31"))
    fig.show()


## Multi-scenario CF comparison (all weather years)

Sources dict keys: `"{year} – Damage-adjusted CF × 0.616"` and `"{year} – Actual CF"`.
- **Colour** encodes weather year.
- **Symbol** encodes data type: `circle` = Actual CF, `x` = Damage-adjusted.
- Pass `date_range=("MM-DD", "MM-DD")` to compare the same calendar period across all years.

In [39]:
# Build unified source dict across all discovered scenarios
multi_sources = build_multi_scenario_sources(scenarios, cf_fixed_factor=cf_fixed_factor)
list(multi_sources.keys())

['2018 – Damage-adjusted CF × 0.616',
 '2018 – Actual CF',
 '2019 – Damage-adjusted CF × 0.616',
 '2019 – Actual CF',
 '2020 – Damage-adjusted CF × 0.616',
 '2020 – Actual CF',
 '2021 – Damage-adjusted CF × 0.616',
 '2021 – Actual CF',
 '2022 – Damage-adjusted CF × 0.616',
 '2022 – Actual CF']

In [40]:
# Single plant — all weather years, summer period
# date_range MM-DD applies July independently to each year
fig = cf_to_temp_max(
    cutout_data_by_year, powerplants_df, multi_sources,
    plant_name="St Alban",
    date_range=("07-01", "07-31"),
)
fig.show()

In [45]:
actual_sources = {k: v for k, v in multi_sources.items() if "Actual" in k}

# All plants — loop over each, all weather years
for plant in scenarios[0]["damage_df"].columns:
    print(plant)
    fig = cf_to_temp_max(
        cutout_data_by_year, powerplants_df, actual_sources,
        plant_name=plant,
        date_range=("06-01", "08-31"),
    )
    fig.show()

Nogent


Paluel 2


Penly 3


St Alban


Dampierre 4


...

Chooz 2


Civaux


Cruas 3


Flamanville 2


Golfech 2


actual_sources = {k: v for k, v in multi_sources.items() if "Actual" in k}

for plant in scenarios[0]["damage_df"].columns:
    print(plant)
    fig = cf_dev_to_temp_p90(
        cutout_data_by_year, powerplants_df, actual_sources,
        plant_name=plant,
        date_range=("06-01", "08-31"),
    )
    fig.show()
